In [39]:
import os
import json
import tempfile
from dotenv import load_dotenv
import pandas as pd 
import gradio as gr 
from openai import OpenAI
from anthropic import Anthropic

In [41]:
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

openai = OpenAI(api_key=openai_api_key)
anthropic = Anthropic(api_key=anthropic_api_key)



OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-


In [42]:
OPENAI_MODEL = "gpt-5.4-mini"
CLAUDE_MODEL = "claude-haiku-4-5"

## Schema Function

In [44]:

def create_schema(description):

    system_prompt = "You design clean schemas for synthetic datasets."

    user_prompt = f"""
you are a data schema designer.
The user wants to generate this synthentic dataset: 
{description}
Design a useful schema for this dataset.
Return ONLY valid JSON in this format:
{{
    "dataset_name:" "string",
    "description": "string",
    "columns": [
        {{
            "name": "column_name",
            "type": "string|integer|number|boolean",
            "description": "what this field represents"
        }}
    ]
}}

Rules:

- Create between 5 and 10 columns.
- Use realistic columns.
- Include a target variable if appropriate.
- Use only these types:
  string
  integer
  number
  boolean
- Do not create unnecessary columns.
"""
    messages = [{"role": "system", "content": system_prompt},{"role": "user", "content": user_prompt}]
    response = openai.chat.completions.create(model = OPENAI_MODEL, messages=messages, response_format={"type": "json_object"}, temperature=0.4)
  
    return json.loads(response.choices[0].message.content)


In [45]:
description = """
Create a synthetic dataset for predicting telecom customer churn.
Include customer demographics, billing information and usage behaviour.
"""

schema = create_schema(description)

print(json.dumps(schema, indent = 2))

{
  "dataset_name": "telecom_customer_churn",
  "description": "Synthetic dataset for predicting whether a telecom customer will churn, including customer demographics, billing information, and usage behavior.",
  "columns": [
    {
      "name": "customer_id",
      "type": "string",
      "description": "Unique identifier for each customer"
    },
    {
      "name": "age",
      "type": "integer",
      "description": "Customer age in years"
    },
    {
      "name": "tenure_months",
      "type": "integer",
      "description": "Number of months the customer has been with the telecom provider"
    },
    {
      "name": "monthly_charge",
      "type": "number",
      "description": "Current monthly billing amount for the customer"
    },
    {
      "name": "total_data_usage_gb",
      "type": "number",
      "description": "Total mobile or internet data used by the customer in gigabytes"
    },
    {
      "name": "support_calls_last_3_months",
      "type": "integer",
      "des

## Generating Data set with OpenAI function

In [ ]:
def generate_with_openai(schema,num_rows):

    schema_text = json.dumps(schema, indent = 2)

    system_prompt = "You generate realistic synthetic tabular data."
    user_prompt = f"""
Generate {num_rows=} realistic synthetic records.
Dataset schema: {schema_text}
Requirements: 
- Return ONLY valid JSON.
- Return an object with a "rows" key. 
- "rows" must contain exactly {num_rows} records.
- Follow the column nammes and data types.
- Values should be realistic.
- Do not use null values.
- Avoid duplicate records.
- Create realistic relationships between fields.
- Make the records diverse. 
Format:
{{
    "rows": [
    {{
        "column_name": "value"
    }}
    ]
}}
"""
    messages = [{"role": "system", "content": system_prompt},{"role": "user", "content": user_prompt}]
    response = openai.chat.completions.create(model = OPENAI_MODEL, messages = messages,response_format={"type": "json_object"}, temperature= 0.8)
    
    data = json.loads(response.choices[0].message.content)

    return data["rows"]

## Generating Data set with Claude function

In [107]:
from pandas.core.arrays import categorical


def generate_with_claude(schema, num_rows):

    schema_text = json.dumps(schema, indent = 2)

    system_prompt = "You are a synthetic data generation specialist."

    user_prompt = f"""
Generate {num_rows} realistic records using this schema:
{schema_text}
Return ONLY valid JSON:
{{
    "rows": [
    {{
        "column_name": "value"
    }}
    ]
}}
Requirments:
- Generate exactly {num_rows} records.
- Follow the schema exactly.
- Avoid duplicates.
- Make the records diverse. 
- Create realistic relationships between variables. 
- Do not use Markdown code fences.
- Do not generate null values. 
- Vary categorical and numerical values naturally. 
- Avoid making every record look similar. 
"""

    response = anthropic.messages.create(model = CLAUDE_MODEL, max_tokens = 12000, temperature=0.8,system= system_prompt, messages= [{"role": "user", "content": user_prompt}])
    
    text = response.content[0].text.strip()

    if text.startswith("```"):
        text = text.strip("`")
        if text.startswith("json"):
            text = text[len("json"):]
        text = text.strip()

    try:
        data = json.loads(text)
    except json.JSONDecodeError as e:
        raise ValueError(
            f"Claude did not return valid JSON.\n"
            f"stop_reason={response.stop_reason}\n"
            f"First 300 chars: {text[:300]}"
        ) from e


    return data["rows"]

## Testing both generators

In [70]:
openai_data = generate_with_openai(schema,10)
claude_data = generate_with_claude(schema,10)

print("OpenAI records:" , len(openai_data))
print("Claude records:", len(claude_data))

GPT
{"rows":[{"customer_id":"CUST-1001","age":24,"tenure_months":4,"monthly_charge":79.99,"total_data_usage_gb":68.4,"support_calls_last_3_months":3,"contract_type":"month-to-month","churned":true},{"customer_id":"CUST-1002","age":41,"tenure_months":36,"monthly_charge":54.5,"total_data_usage_gb":124.7,"support_calls_last_3_months":0,"contract_type":"annual","churned":false},{"customer_id":"CUST-1003","age":58,"tenure_months":8,"monthly_charge":92.3,"total_data_usage_gb":42.1,"support_calls_last_3_months":4,"contract_type":"month-to-month","churned":true},{"customer_id":"CUST-1004","age":33,"tenure_months":18,"monthly_charge":68.0,"total_data_usage_gb":210.5,"support_calls_last_3_months":1,"contract_type":"two-year","churned":false},{"customer_id":"CUST-1005","age":29,"tenure_months":12,"monthly_charge":45.99,"total_data_usage_gb":95.2,"support_calls_last_3_months":2,"contract_type":"annual","churned":false},{"customer_id":"CUST-1006","age":67,"tenure_months":2,"monthly_charge":88.75,"t

In [71]:
openai_data[:2]

[{'customer_id': 'CUST-1001',
  'age': 24,
  'tenure_months': 4,
  'monthly_charge': 79.99,
  'total_data_usage_gb': 68.4,
  'support_calls_last_3_months': 3,
  'contract_type': 'month-to-month',
  'churned': True},
 {'customer_id': 'CUST-1002',
  'age': 41,
  'tenure_months': 36,
  'monthly_charge': 54.5,
  'total_data_usage_gb': 124.7,
  'support_calls_last_3_months': 0,
  'contract_type': 'annual',
  'churned': False}]

In [72]:
claude_data[:2]

[{'customer_id': 'CUST001847',
  'age': 34,
  'tenure_months': 24,
  'monthly_charge': 65.99,
  'total_data_usage_gb': 156.3,
  'support_calls_last_3_months': 2,
  'contract_type': 'annual',
  'churned': False},
 {'customer_id': 'CUST002156',
  'age': 67,
  'tenure_months': 8,
  'monthly_charge': 45.5,
  'total_data_usage_gb': 12.7,
  'support_calls_last_3_months': 5,
  'contract_type': 'month-to-month',
  'churned': True}]

## Combine into Pandas

In [73]:
def combine_data(openai_data, claude_data, schema):
    all_rows = openai_data + claude_data

    dataframe = pd.DataFrame(all_rows)

    column_order = [
        column["name"]
        for column in schema["columns"]
    ]
    existing_columns = [
        column
        for column in column_order
        if column in dataframe.columns
    ]

    dataframe = dataframe[existing_columns]

    return dataframe

In [74]:
df = combine_data(openai_data,claude_data,schema)

df

,customer_id,age,tenure_months,monthly_charge,total_data_usage_gb,support_calls_last_3_months,contract_type,churned
0,CUST-1001,24,4,79.99,68.4,3,month-to-month,True
1,CUST-1002,41,36,54.50,124.7,0,annual,False
2,CUST-1003,58,8,92.30,42.1,4,month-to-month,True
3,CUST-1004,33,18,68.00,210.5,1,two-year,False
4,CUST-1005,29,12,45.99,95.2,2,annual,False
5,CUST-1006,67,2,88.75,18.6,5,month-to-month,True
6,CUST-1007,46,54,72.40,340.8,1,two-year,False
7,CUST-1008,22,6,39.95,56.9,1,month-to-month,False
8,CUST-1009,52,27,84.20,154.3,2,annual,False
9,CUST-1010,38,3,96.50,12.4,6,month-to-month,True


## Validation

In [ ]:
def validate_dataframe (dataframe, schema):
    expected_columns = [
        column['name']
        for column in schema["columns"]
    ]

    missing_columns = [
        column
        for column in expected_columns
        if column not in dataframe.columns
    ]

    duplicate_rows = int(
        dataframe.duplicated().sum()
    )

    missing_values = int(
        dataframe.isnull().sum().sum()
    )

    return {
        "rows": len(dataframe),
        "columns": len(dataframe.columns),
        "missing_columns": missing_columns,
        "duplicate_rows": duplicate_rows,
        "missing_values": missing_values
    }



In [76]:
validation = validate_dataframe(df,schema)

validation

{'rows': 20,
 'columns': 8,
 'missing_columns': [],
 'duplicate_rows': 0,
 'missing_values': 0}

## Critic Agent Function

In [93]:
def critique_dataset(schema, dataframe):

    sample = dataframe.head(20).to_dict(
        orient= "records"
    )

    system_prompt = "You are a data quality critic."

    user_prompt = f"""
We generated a synthetic dataset using this schema:
{json.dumps(schema, indent=2)}
Here is the sample of the generated data:
{json.dumps(sample, indent=2)}
Return ONLY valid JSON:

{{
    "quality_score": 0,
    "issues": [],
    "strengths": [],
    "recommendations": ""
}}

{{
    "quality_score": 85,
    "issues": [
        "Some customers have unusually high data usage for their contract type."
    ],
    "strengths": [
        "No missing values were observed.",
        "The categorical values are consistent.",
        "The dataset contains reasonable variation."
    ],
    "recommendations": "Consider increasing variation in customer tenure and monthly charges."
}}


Requirements:

- Do not use Markdown code fences.
- Do not include any explanation before or after the JSON.
- Quality score must be between 0 and 100.

Check for:

- incorrect data types
- unrealistic values
- duplicate-looking records
- inconsistent categories
- suspicious relationships 
- lack of diversity 
- missing values
- obvious generation artifacts

Give a quality score from 0 to 100
"""
    messages = [{"role":"user", "content":user_prompt}]
    response = anthropic.messages.create(model = CLAUDE_MODEL, max_tokens=2000, system= system_prompt,messages=messages)

    text = response.content[0].text.strip()
    if text.startswith("```"):
       text = text.strip("`").removeprefix("json").strip()
    try:
       return json.loads(text)
    except json.JSONDecodeError as e:
       raise ValueError(f"Model did not return valid JSON: {text[:200]}") from e

In [ ]:
critique = critique_dataset(schema,df)

In [95]:
critique

{'quality_score': 72,
 'issues': ["Inconsistent customer_id formatting: mix of 'CUST-####' and 'CUST######' patterns suggests incomplete standardization",
  'Suspicious correlation patterns: churned customers consistently have low tenure (2-15 months) and high support calls (3-7), while non-churned have high tenure and low support calls - too perfectly separated',
  'Unrealistic data usage variance: same age/tenure cohorts show extreme variation (e.g., CUST010456 with 521.2 GB vs CUST008567 with 203.7 GB, both similar profiles)',
  'Potential generation artifact: churned=true strongly correlates with month-to-month contracts (10/10 churned samples) and support_calls >= 3, suggesting rule-based generation rather than realistic patterns',
  'Missing realistic edge cases: no customers with very high tenure and churn, or low support calls with churn'],
 'strengths': ['No missing values across all records',
  'All data types match schema correctly',
  'Age range (22-67) is realistic for tel

## Complete Pipeline 

In [96]:
def generate_dataset(description, num_rows = 50):
    
    print("Creating schema...")
    schema = create_schema(description)
    print(f"Created {len(schema['columns'])} columns")

    openai_rows = num_rows // 2 
    claude_rows = num_rows - openai_rows 

    print(f"OpenAI generating {openai_rows} records...")
    openai_data = generate_with_openai(schema,openai_rows)

    print(f"Claude generating {claude_rows} records...")
    claude_data = generate_with_claude(schema,claude_rows)

    print("Combining datasets...")
    dataframe = combine_data(openai_data,claude_data,schema)

    print("Running validation...")
    validation = validate_dataframe(dataframe,schema)

    print("Critic reviewing dataset...")
    critique = critique_dataset(schema, dataframe)

    print("Generation complete!")

    return{
        "schema": schema,
        "dataframe": dataframe,
        "validation": validation,
        "critique": critique
    }


## Testing the entire pipeline 

In [97]:
description = """ 
Create a synthetic customer churn dataset
for a telecommunications company.

Include customer demographics, billing information,
service usage and whether the customer churned.
"""

result = generate_dataset(description, num_rows=50)

Creating schema...
Created 8 columns
OpenAI generating 25 records...
GPT
{"rows":[{"customer_age":24,"gender":"Female","tenure_months":3,"monthly_charges":68.4,"total_data_usage_gb":142.7,"num_support_calls":2,"contract_type":"Month-to-month","churned":true},{"customer_age":57,"gender":"Male","tenure_months":48,"monthly_charges":84.9,"total_data_usage_gb":310.2,"num_support_calls":1,"contract_type":"Two-year","churned":false},{"customer_age":36,"gender":"Female","tenure_months":15,"monthly_charges":59.3,"total_data_usage_gb":96.8,"num_support_calls":0,"contract_type":"One-year","churned":false},{"customer_age":29,"gender":"Male","tenure_months":6,"monthly_charges":92.1,"total_data_usage_gb":420.5,"num_support_calls":4,"contract_type":"Month-to-month","churned":true},{"customer_age":44,"gender":"Female","tenure_months":22,"monthly_charges":73.8,"total_data_usage_gb":188.4,"num_support_calls":1,"contract_type":"One-year","churned":false},{"customer_age":63,"gender":"Male","tenure_months"

In [98]:
result["dataframe"]

,customer_age,gender,tenure_months,monthly_charges,total_data_usage_gb,num_support_calls,contract_type,churned
0,24,Female,3,68.40,142.7,2,Month-to-month,True
1,57,Male,48,84.90,310.2,1,Two-year,False
2,36,Female,15,59.30,96.8,0,One-year,False
3,29,Male,6,92.10,420.5,4,Month-to-month,True
4,44,Female,22,73.80,188.4,1,One-year,False
5,63,Male,72,101.60,512.9,0,Two-year,False
6,31,Female,9,64.70,121.3,3,Month-to-month,True
7,52,Male,36,88.20,276.1,1,One-year,False
8,27,Female,12,54.60,84.5,0,One-year,False
9,40,Male,5,79.50,233.8,5,Month-to-month,True


In [99]:
print(json.dumps(result["schema"], indent = 2))

{
  "dataset_name": "telecom_customer_churn",
  "description": "Synthetic customer churn dataset for a telecommunications company, including customer demographics, billing information, service usage, and a churn target.",
  "columns": [
    {
      "name": "customer_age",
      "type": "integer",
      "description": "Age of the customer in years"
    },
    {
      "name": "gender",
      "type": "string",
      "description": "Customer gender"
    },
    {
      "name": "tenure_months",
      "type": "integer",
      "description": "Number of months the customer has been with the company"
    },
    {
      "name": "monthly_charges",
      "type": "number",
      "description": "Total monthly billing amount for the customer"
    },
    {
      "name": "total_data_usage_gb",
      "type": "number",
      "description": "Total monthly data usage in gigabytes"
    },
    {
      "name": "num_support_calls",
      "type": "integer",
      "description": "Number of customer support calls 

In [100]:
result["validation"]

{'rows': 50,
 'columns': 8,
 'missing_columns': [],
 'duplicate_rows': 0,
 'missing_values': 0}

In [ ]:
print(json.dumps(result["critique"], indent = 2))

{
  "quality_score": 72,
  "issues": [
    "Strong artificial correlation between tenure_months and monthly_charges - longer tenure consistently results in higher charges, which is unrealistic for telecom billing",
    "Perfect correlation between num_support_calls and churned status - customers with 3+ support calls always churn, suggesting oversimplified synthetic generation",
    "Data usage values appear suspiciously high relative to typical mobile/telecom usage patterns (up to 610GB monthly is unrealistic for individual consumers)",
    "Monthly charges show artificial banding around specific price points ($49.9-$112.3) with limited realistic variation",
    "Gender distribution appears perfectly balanced (10 Female, 10 Male in sample) suggesting synthetic uniformity rather than real-world variation",
    "Churn rate of 40% in sample (8/20) is notably high and may not reflect realistic telecom industry rates"
  ],
  "strengths": [
    "No missing values present in any field",
    

## Gradio 

In [113]:
def gradio_generate(description, num_rows):

    result = generate_dataset(description, int(num_rows))

    schema = result["schema"]
    dataframe = result["dataframe"]
    validation = result["validation"]
    critique = result ["critique"]

    quality_report = f"""
## Dataset Quality

**AI Quality Score:** {critique["quality_score"]}/100

**Rows:** {validation["rows"]}

**Columns:** {validation["columns"]}

**Missing Values:** {validation["missing_values"]}
    
**Duplicate Rows:** {validation["duplicate_rows"]}

### Strengths

{chr(10).join("- " + x for x in critique["strengths"])}

### Issues

{chr(10).join("- " + x for x in critique["issues"])}

### Recommendation

{critique["recommendations"]}
"""

    schema_text = json.dumps(schema, indent=2)
    return dataframe, schema_text, quality_report

In [114]:
with gr.Blocks(title = "Synthetic Data Foundry") as generator:

    gr.Markdown(
        """
# Synthetic Data Foundry 

### Multi-Agent Synthetic Dataset Generator

Generate synthetic datasets using OpenAI and Claude.
"""
    )

    with gr.Tab("Generate"):
        description = gr.Textbox(
            label = "Dataset Description",
            placeholder = "Example: Generate a telecom customer churn dataset...",
            lines = 5
        )
        num_rows = gr.Slider(minimum=10, maximum=100, value=50, step=10, label="Number of Records")
        generate_button = gr.Button("Generate Dataset", variant = "primary")
    
    with gr.Tab("Dataset"):
        dataframe_output = gr.Dataframe(label="Generate Dataset", interactive=False)

    with gr.Tab("Schema"):
        schema_output = gr.Code(label="Generated Schema", language="json")

    with gr.Tab("Quality Report"):
        quality_output = gr.Markdown("Generate a dataset to see the quality report.")

    generate_button.click(
        fn = gradio_generate,
        inputs = [description, num_rows],
        outputs = [dataframe_output, schema_output, quality_output]
    )

In [116]:
generator.launch(inbrowser= True )

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
* To create a public link, set `share=True` in `launch()`.


Creating schema...
Created 8 columns
OpenAI generating 25 records...
GPT
{"rows":[{"age":78,"sex":"Female","systolic_blood_pressure":156.4,"heart_rate":92,"chronic_condition_count":5,"medication_count":11,"previous_hospital_stay_length_days":9,"readmitted_within_30_days":true},{"age":64,"sex":"Male","systolic_blood_pressure":148.2,"heart_rate":88,"chronic_condition_count":4,"medication_count":9,"previous_hospital_stay_length_days":7,"readmitted_within_30_days":true},{"age":49,"sex":"Female","systolic_blood_pressure":132.7,"heart_rate":79,"chronic_condition_count":2,"medication_count":5,"previous_hospital_stay_length_days":3,"readmitted_within_30_days":false},{"age":83,"sex":"Male","systolic_blood_pressure":164.9,"heart_rate":96,"chronic_condition_count":6,"medication_count":12,"previous_hospital_stay_length_days":12,"readmitted_within_30_days":true},{"age":57,"sex":"Female","systolic_blood_pressure":141.3,"heart_rate":84,"chronic_condition_count":3,"medication_count":7,"previous_hospit